### 1. Data Loading and Exploration:
* Utilize Pandas to load the dataset and explore its initial structure.
* Summarize features, target variable, and their respective data types.
* Conduct basic descriptive statistics for an overview of the dataset.


In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
mobile_data = pd.read_csv('train.csv')

mobile_data.head()

In [ ]:
mobile_data.info()

In [ ]:
mobile_data.describe()

### 2. Data Cleaning and Preprocessing:
* Address missing or null values.
* Transform categorical data into numerical format using suitable methods.

In [ ]:
mobile_data.isnull().sum()

### 3. Statistical Analysis with NumPy and SciPy:
* Execute detailed statistical analysis on each feature, including:
    * Calculation of central tendency measures (mean, median, mode).
    * Analysis of variability (range, variance, standard deviation).
    * Evaluation of distribution shapes through skewness and kurtosis.
* Perform hypothesis testing for statistical significance between groups (e.g., different price ranges).
* Investigate feature-target correlations using SciPy.
* Apply advanced SciPy statistical functions for deeper insights.

In [ ]:
mean = mobile_data.mean()
median = mobile_data.median()
mode = mobile_data.mode().iloc[0]

range_val = mobile_data.max() -  mobile_data.min()
variance = mobile_data.var()
std_dev = mobile_data.std()

skewness = mobile_data.skew()
kurtosis = mobile_data.kurtosis()

stats_summary = pd.DataFrame({
    'Mean': mean, 
    'Median': median,
    'Mode': mode, 
    'Range': range_val,
    'Variance': variance,
    'Std Dev': std_dev,
    'Skewness': skewness,
    'Kurtosis': kurtosis
})

print(stats_summary)

In [ ]:
# List of features to test against price range
features_to_test = ['ram', 'battery_power', 'px_width', 'px_height', 'mobile_wt', 'blue', 'clock_speed', 
                    'dual_sim', 'fc', 'four_g', 'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'sc_h', 'sc_w', 'talk_time', 'three_g', 'touch_screen', 'wifi']

print("--- ANOVA Results (Testing relationship with Price Range) ---")

for feature in features_to_test:
    # Group feature data by the 4 price ranges
    groups = [mobile_data[mobile_data['price_range'] ==i][feature] for i in range(4)]

    f_stat, p_val = stats.f_oneway(*groups)

    status = "Significant" if p_val < 0.05 else "Not Significant"
    print(f"{feature:15} | P-Value: {p_val:.4e} | Result: {status}")

* H0: For each feature, it does not have an impact on the price range
* H1: For each feature, it does have an impact on price range

Result: Each of the significant (p-value less than 0.05) vary significantly across different price points.

In [ ]:
# Spearman Rank Correlation 

# Correlation between all numeric features and the target (price range)
results = []
for col in mobile_data.columns:
    if col != 'price_range':
        coef, p_val = stats.spearmanr(mobile_data[col], mobile_data['price_range'])
        results.append({'Feature': col, 'Correlation': coef, 'P-Value': p_val})

# Convert to DF and sort by strongest correlation
corr_df = pd.DataFrame(results).sort_values(by='Correlation', ascending=False)
print(corr_df.head(5))

### 4. Data Visualization with Matplotlib:
* Produce histograms, scatter plots, and box plots for data distribution and relationship insights.
* Employ heatmaps for correlation visualization.
* Ensure clarity in plots with appropriate titles, labels, and axis information.

In [ ]:
# Histogram 
sns.histplot(mobile_data['battery_power'], kde=True, color='skyblue')
plt.title('Distribution of Battery Power')
plt.show()


In [ ]:
# Group comparison using box plots
sns.boxplot(x='price_range', y='ram', data=mobile_data, hue='price_range', legend=False, palette='viridis')
plt.title('RAM Distribution by Price Range')
plt.show()

In [ ]:
# Scatter plot to show the relationship between features

sns.scatterplot(x='ram', y='battery_power', hue='price_range', data=mobile_data)
plt.title('RAM vs Battery Power by Price Range')
plt.show()

### 5. Insight Synthesis and Conclusion:
* Derive conclusions from statistical tests and visualizations.
* Identify key determinants in mobile price classification.
* Highlight any unexpected or significant findings.

1. We can confirm that price range is not random. This is seen in the one-way ANOVA and in the Spearman Correlations. 
2. The box plots reveal that as the price range increases, the median values for key features (especially RAM) increases as well 
3. The histogram for battery_power shows uniform distribution, meaning the dataset is balanced of both low end and high end hardware. 

Key Determinants of Price
1. RAM: With the correlation often exceeding 0.90, RAM is the deal-breaker, showing the clearest separation between all four price categories
2. Battery Power: A Secondary determinant. It is the boost that will move a device into a higher bracket
3. Pixel resolution (height and width): statistical tests showed these are significant, indicating that screen quality is a premium feature 

Unexepected / Significant Findings 
1. Feature interaction exists - we see this in the slanted effect of the scatter-plot. A phone having just RAM doesn't make it expensive. A combination of high RAM and high batter power is what will define the very high cost (3) category
2. features like clock_speed and n_cores showed very low correlation. Consumers value capacity and display more than raw processing speed or physical dimensions when determining price. 